In [2]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))

from src.data_loader import read_csv_from_blob
from src.dataset import assemble_dataset
from src.train import train_per_horizon_models, predict_next_day

STORAGE_ACCOUNT = "sagreekdamdevweu"
GATE_CLOSURE_HOUR = 12
TARGET_DAY_ATHENS = pd.Timestamp("2026-05-15", tz="Europe/Athens")

print(f"Target prediction day (Athens): {TARGET_DAY_ATHENS.date()}")

Target prediction day (Athens): 2026-05-15


In [3]:
print("Loading processed/ data from blob...")

dam = read_csv_from_blob(STORAGE_ACCOUNT, "processed", "dam_prices/full.csv")
load = read_csv_from_blob(STORAGE_ACCOUNT, "processed", "load_forecast/full.csv")
renewable = read_csv_from_blob(STORAGE_ACCOUNT, "processed", "renewable_forecast/full.csv")

print(f"DAM:        {len(dam)} rows, {dam.index.min()} → {dam.index.max()}")
print(f"Load:       {len(load)} rows, {load.index.min()} → {load.index.max()}")
print(f"Renewable:  {len(renewable)} rows, {renewable.index.min()} → {renewable.index.max()}")

Loading processed/ data from blob...


DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	WorkloadIdentityCredential: WorkloadIdentityCredential authentication unavailable. The workload options are not fully configured. See the troubleshooting guide for more information: https://aka.ms/azsdk/python/identity/workloadidentitycredential/troubleshoot. Missing required arguments: 'tenant_id', 'client_id', 'token_file_path'.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	VisualStudioCodeCredential: VisualStudioCodeCredential requires the 'azure-identity-broker' pa

DAM:        29520 rows, 2022-12-31 22:00:00+00:00 → 2026-05-14 21:00:00+00:00
Load:       29543 rows, 2022-12-31 22:00:00+00:00 → 2026-05-15 20:00:00+00:00
Renewable:  29543 rows, 2022-12-31 22:00:00+00:00 → 2026-05-15 20:00:00+00:00


In [4]:
renewable

,solar,wind_onshore
2022-12-31 22:00:00+00:00,0.0,495.0
2022-12-31 23:00:00+00:00,0.0,518.0
2023-01-01 00:00:00+00:00,0.0,560.0
2023-01-01 01:00:00+00:00,0.0,603.0
2023-01-01 02:00:00+00:00,0.0,649.0
...,...,...
2026-05-15 16:00:00+00:00,780.0,1492.5
2026-05-15 17:00:00+00:00,52.5,1465.0
2026-05-15 18:00:00+00:00,0.0,1407.5
2026-05-15 19:00:00+00:00,0.0,1317.5


In [5]:
prices_train, exog_train = assemble_dataset(dam, load, renewable, join="inner")
print(f"Training data (inner join): {len(prices_train)} rows")
print(f"Range: {prices_train.index.min()} → {prices_train.index.max()}")

Training data (inner join): 29520 rows
Range: 2022-12-31 22:00:00+00:00 → 2026-05-14 21:00:00+00:00


In [6]:
prices_train, exog_train = assemble_dataset(dam, load, renewable, join="inner")
print(f"Training data (inner join): {len(prices_train)} rows")
print(f"Range: {prices_train.index.min()} → {prices_train.index.max()}")

lgbm_params = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_child_samples": 20,
    "random_state": 42,
    "verbose": -1,
}

result = train_per_horizon_models(
    prices_train,
    exog=exog_train,
    gate_closure_hour=GATE_CLOSURE_HOUR,
    horizons=tuple(range(0, 24)),
    same_hour_lag_days=(1, 2, 7),
    context_window=1,
    test_days=0,  # ← FULL DATASET
    lgbm_params=lgbm_params,
)

print("\nTraining complete!")
print(f"Models trained: {len(result.models)}")
print(f"(Metrics are NaN since no test set)")


Training data (inner join): 29520 rows
Range: 2022-12-31 22:00:00+00:00 → 2026-05-14 21:00:00+00:00

Training complete!
Models trained: 24
(Metrics are NaN since no test set)


In [9]:
################################
# Forecast for May 14
################################

prices, exog = assemble_dataset(dam, load, renewable, join="outer")

# Forecast time = noon Athens on May 13 (today)
forecast_time_athens = TARGET_DAY_ATHENS - pd.Timedelta(days=1) + pd.Timedelta(hours=GATE_CLOSURE_HOUR)
forecast_time_utc = forecast_time_athens.tz_convert("UTC")
print(f"Forecast time (Athens): {forecast_time_athens}")
print(f"Forecast time (UTC):    {forecast_time_utc}")

if forecast_time_utc not in prices.index:
    print(f"WARNING: forecast_time not in prices index!")
    closest = prices.index[prices.index <= forecast_time_utc][-1]
    print(f"Closest available: {closest}")
else:
    print(f"forecast_time present in index ✓")

forecast_may14 = predict_next_day(
    result, prices, exog=exog, forecast_time=forecast_time_utc
)

print(f"\nForecast for May 15:")
print(forecast_may14.to_frame(name="predicted_eur_mwh"))

Forecast time (Athens): 2026-05-14 12:00:00+03:00
Forecast time (UTC):    2026-05-14 09:00:00+00:00
forecast_time present in index ✓

Forecast for May 15:
                           predicted_eur_mwh
2026-05-14 21:00:00+00:00         128.955652
2026-05-14 22:00:00+00:00         115.148304
2026-05-14 23:00:00+00:00         109.636350
2026-05-15 00:00:00+00:00         114.390184
2026-05-15 01:00:00+00:00         117.009668
2026-05-15 02:00:00+00:00         116.769231
2026-05-15 03:00:00+00:00         142.054572
2026-05-15 04:00:00+00:00         134.727763
2026-05-15 05:00:00+00:00         101.006384
2026-05-15 06:00:00+00:00          63.638822
2026-05-15 07:00:00+00:00           8.363561
2026-05-15 08:00:00+00:00           2.135455
2026-05-15 09:00:00+00:00          -5.565712
2026-05-15 10:00:00+00:00           6.624517
2026-05-15 11:00:00+00:00          40.665863
2026-05-15 12:00:00+00:00          79.693440
2026-05-15 13:00:00+00:00         109.901415
2026-05-15 14:00:00+00:00         1

In [10]:
# Fetch published DAM for May 14 (Athens)
from src.data_loader import fetch_dam_prices
from src.processing import ensure_hourly

print("Fetching published DAM for May 14 from ENTSO-E...")

# Fetch a window covering May 14 (Athens local)
start = pd.Timestamp("2026-05-15", tz="Europe/Athens")
end = pd.Timestamp("2026-05-17", tz="Europe/Athens")

actual_may14 = fetch_dam_prices(start=start, end=end, country_code="GR")
actual_may14 = ensure_hourly(actual_may14)

# Convert index to UTC so it matches our forecast format
actual_may14.index = actual_may14.index.tz_convert("UTC")

print(f"Fetched {len(actual_may14)} rows")
print(f"Range: {actual_may14.index.min()} → {actual_may14.index.max()}")
print()
print(actual_may14.to_string())

Fetching published DAM for May 14 from ENTSO-E...
Fetched 25 rows
Range: 2026-05-14 21:00:00+00:00 → 2026-05-15 21:00:00+00:00

2026-05-14 21:00:00+00:00    135.9900
2026-05-14 22:00:00+00:00    133.6000
2026-05-14 23:00:00+00:00    126.2275
2026-05-15 00:00:00+00:00    123.4375
2026-05-15 01:00:00+00:00    124.4975
2026-05-15 02:00:00+00:00    131.9225
2026-05-15 03:00:00+00:00    134.7600
2026-05-15 04:00:00+00:00    142.1900
2026-05-15 05:00:00+00:00    127.6950
2026-05-15 06:00:00+00:00    106.9700
2026-05-15 07:00:00+00:00      0.0075
2026-05-15 08:00:00+00:00      0.0000
2026-05-15 09:00:00+00:00      0.0000
2026-05-15 10:00:00+00:00      0.0000
2026-05-15 11:00:00+00:00      0.0000
2026-05-15 12:00:00+00:00      0.0000
2026-05-15 13:00:00+00:00      0.0000
2026-05-15 14:00:00+00:00     26.0125
2026-05-15 15:00:00+00:00    127.5225
2026-05-15 16:00:00+00:00    141.2375
2026-05-15 17:00:00+00:00    157.3425
2026-05-15 18:00:00+00:00    165.6000
2026-05-15 19:00:00+00:00    147.080

In [11]:
print("Fetching published DAM for May 14 from ENTSO-E...")

# Fetch a window covering May 14 (Athens local)
start = pd.Timestamp("2026-05-14", tz="Europe/Athens")
end = pd.Timestamp("2026-05-15", tz="Europe/Athens")

actual_may13 = fetch_dam_prices(start=start, end=end, country_code="GR")
actual_may13 = ensure_hourly(actual_may13)

# Convert index to UTC so it matches our forecast format
actual_may13.index = actual_may13.index.tz_convert("UTC")

print(f"Fetched {len(actual_may13)} rows")
print(f"Range: {actual_may13.index.min()} → {actual_may13.index.max()}")
print()
print(actual_may13.to_string())

Fetching published DAM for May 14 from ENTSO-E...
Fetched 25 rows
Range: 2026-05-13 21:00:00+00:00 → 2026-05-14 21:00:00+00:00

2026-05-13 21:00:00+00:00    160.9650
2026-05-13 22:00:00+00:00    126.2850
2026-05-13 23:00:00+00:00    123.5500
2026-05-14 00:00:00+00:00    123.1750
2026-05-14 01:00:00+00:00    131.3450
2026-05-14 02:00:00+00:00    136.3950
2026-05-14 03:00:00+00:00    136.9575
2026-05-14 04:00:00+00:00    141.9975
2026-05-14 05:00:00+00:00    121.2575
2026-05-14 06:00:00+00:00    107.9600
2026-05-14 07:00:00+00:00     26.2475
2026-05-14 08:00:00+00:00      0.0100
2026-05-14 09:00:00+00:00      0.0375
2026-05-14 10:00:00+00:00      0.0125
2026-05-14 11:00:00+00:00      0.0100
2026-05-14 12:00:00+00:00      0.0050
2026-05-14 13:00:00+00:00      0.0125
2026-05-14 14:00:00+00:00     76.3125
2026-05-14 15:00:00+00:00    111.2525
2026-05-14 16:00:00+00:00    158.3675
2026-05-14 17:00:00+00:00    155.6300
2026-05-14 18:00:00+00:00    172.4325
2026-05-14 19:00:00+00:00    142.990

In [12]:
naive = actual_may13
naive.index = naive.index + pd.Timedelta(days=1)
naive

2026-05-14 21:00:00+00:00    160.9650
2026-05-14 22:00:00+00:00    126.2850
2026-05-14 23:00:00+00:00    123.5500
2026-05-15 00:00:00+00:00    123.1750
2026-05-15 01:00:00+00:00    131.3450
2026-05-15 02:00:00+00:00    136.3950
2026-05-15 03:00:00+00:00    136.9575
2026-05-15 04:00:00+00:00    141.9975
2026-05-15 05:00:00+00:00    121.2575
2026-05-15 06:00:00+00:00    107.9600
2026-05-15 07:00:00+00:00     26.2475
2026-05-15 08:00:00+00:00      0.0100
2026-05-15 09:00:00+00:00      0.0375
2026-05-15 10:00:00+00:00      0.0125
2026-05-15 11:00:00+00:00      0.0100
2026-05-15 12:00:00+00:00      0.0050
2026-05-15 13:00:00+00:00      0.0125
2026-05-15 14:00:00+00:00     76.3125
2026-05-15 15:00:00+00:00    111.2525
2026-05-15 16:00:00+00:00    158.3675
2026-05-15 17:00:00+00:00    155.6300
2026-05-15 18:00:00+00:00    172.4325
2026-05-15 19:00:00+00:00    142.9900
2026-05-15 20:00:00+00:00    136.9600
2026-05-15 21:00:00+00:00    136.9500
Freq: h, Name: price_eur_mwh, dtype: float64

In [14]:
import plotly.graph_objects as go

# df has a DatetimeIndex and columns: forecast_may14, actual_may14

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=forecast_may14.index,
    y=forecast_may14,
    mode='lines+markers',
    name='forecast_may15',
    line=dict(dash='dash')
))

fig.add_trace(go.Scatter(
    x=actual_may14.index,
    y=actual_may14,
    mode='lines+markers',
    name='actual_may15'
))

fig.add_trace(go.Scatter(
    x=naive.index,
    y=naive,
    mode='lines+markers',
    name='naive'
))

fig.update_layout(
    title='Forecast vs Actual — May 15',
    xaxis_title='Time',
    yaxis_title='Value',
    hovermode='x unified',
    xaxis=dict(tickformat='%H:%M')  # or '%Y-%m-%d %H:%M' for full datetime
)

fig.show()

In [32]:
df = pd.concat([forecast_may14, actual_may14], axis=1)
df = pd.concat([df, naive.rename("naive")], axis=1)
df['error'] = np.abs(df['prediction_eur_mwh'] - df['price_eur_mwh'])
df['naive_error'] = np.abs(df['naive'] - df['price_eur_mwh'])
df

,prediction_eur_mwh,price_eur_mwh,naive,error,naive_error
2026-05-13 21:00:00+00:00,105.098266,160.9650,122.3525,55.866734,38.6125
2026-05-13 22:00:00+00:00,101.164412,126.2850,113.7000,25.120588,12.5850
2026-05-13 23:00:00+00:00,88.233157,123.5500,115.8075,35.316843,7.7425
2026-05-14 00:00:00+00:00,91.788150,123.1750,113.2650,31.386850,9.9100
2026-05-14 01:00:00+00:00,83.252167,131.3450,133.2500,48.092833,1.9050
2026-05-14 02:00:00+00:00,98.031857,136.3950,138.4325,38.363143,2.0375
2026-05-14 03:00:00+00:00,130.174184,136.9575,148.0225,6.783316,11.0650
2026-05-14 04:00:00+00:00,128.061147,141.9975,154.5425,13.936353,12.5450
2026-05-14 05:00:00+00:00,93.326019,121.2575,139.6325,27.931481,18.3750
2026-05-14 06:00:00+00:00,69.444285,107.9600,100.0025,38.515715,7.9575


In [30]:
df['error'].mean()

np.float64(30.117017987023456)

In [33]:
df['naive_error'].mean()

np.float64(11.7554)

In [15]:
forecast_df = forecast_may14.to_frame(name="predicted_price_eur_mwh")
forecast_df.index = forecast_df.index.tz_convert("UTC")
forecast_df.to_csv("local_may14_forecast.csv")
print("Saved local prediction to local_may14_forecast.csv")

Saved local prediction to local_may14_forecast.csv


In [9]:
prices_train

2022-12-31 22:00:00+00:00    210.0500
2022-12-31 23:00:00+00:00    230.9000
2023-01-01 00:00:00+00:00    268.1900
2023-01-01 01:00:00+00:00    229.5800
2023-01-01 02:00:00+00:00    235.9800
                               ...   
2026-05-12 17:00:00+00:00    151.2075
2026-05-12 18:00:00+00:00    125.5650
2026-05-12 19:00:00+00:00    129.7575
2026-05-12 20:00:00+00:00    122.8300
2026-05-12 21:00:00+00:00    122.8000
Freq: h, Name: price_eur_mwh, Length: 29472, dtype: float64

In [13]:
m_tr

NameError: name 'm_tr' is not defined